# 🎓 Chapter 18: p-values Explained Without Jargon
*Track 1 — Students | Tier 2*

> **🎬 Hook:** "p=0.04, therefore our result is real." That sentence contains at least 2 mistakes.

**🎯 Objectives:** Understand what a p-value actually means, and what it doesn't mean.

## 📖 Section 1 — Concept Review

### What a p-value IS:
> The probability of observing results at least as extreme as ours, **assuming H₀ is true**.

$$p = P(\text{data this extreme or more} \mid H_0 \text{ is true})$$

### What a p-value is NOT:
- ❌ NOT the probability that H₀ is true
- ❌ NOT the probability that your result is due to chance
- ❌ NOT the probability that you're wrong
- ❌ NOT a measure of effect size or importance

### Common Misconceptions

| Wrong | Right |
|-------|-------|
| "p=0.03 means 3% chance H₀ is true" | p-value assumes H₀ is true |
| "p=0.06 means no effect" | Might just need more data |
| "p<0.05 means practically important" | Statistical ≠ practical significance |
| "Small p → big effect" | Small p just means unlikely under H₀ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style="whitegrid")
np.random.seed(42)

# ── Visualize p-value ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

mu0 = 0; sigma = 1
x = np.linspace(-4, 4, 300)
y = stats.norm.pdf(x)

observed_z = 2.1
p_val = 2 * (1 - stats.norm.cdf(abs(observed_z)))  # two-tailed

ax = axes[0]
ax.plot(x, y, lw=3, color='#3498db', label='H₀ distribution N(0,1)')
mask_right = x >= abs(observed_z)
mask_left  = x <= -abs(observed_z)
ax.fill_between(x, y, where=mask_right, color='#e74c3c', alpha=0.6, label=f'p-value = {p_val:.4f}')
ax.fill_between(x, y, where=mask_left,  color='#e74c3c', alpha=0.6)
ax.axvline(observed_z, color='#27ae60', lw=3, linestyle='--', label=f'Observed z={observed_z}')
ax.axvline(-observed_z, color='#27ae60', lw=3, linestyle='--')
ax.set_title('p-value = shaded area (two-tailed)', fontweight='bold')
ax.set_xlabel('Z-score'); ax.set_ylabel('Density')
ax.legend(fontsize=9)

# ── Effect of sample size on p-value ──
true_diff = 2  # true difference in means
sample_sizes = [10, 25, 50, 100, 250, 500, 1000]
p_values = []
for n in sample_sizes:
    samples = np.random.normal(true_diff, 10, n)
    t_stat, p = stats.ttest_1samp(samples, 0)
    p_values.append(p)

ax2 = axes[1]
ax2.plot(sample_sizes, p_values, 'bo-', markersize=8, lw=2)
ax2.axhline(0.05, color='red', linestyle='--', lw=2, label='α=0.05')
ax2.set_xscale('log')
ax2.set_xlabel('Sample Size')
ax2.set_ylabel('p-value')
ax2.set_title('⚠️ Same effect, larger n → smaller p-value
(Significance ≠ Importance!)', fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.savefig('ch18_pvalue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# p-hacking demo: run many tests until p<0.05
np.random.seed(123)
n_experiments = 10_000
null_p_values = []

for _ in range(n_experiments):
    # H₀ is TRUE (no effect), but we test anyway
    group_a = np.random.normal(0, 1, 30)
    group_b = np.random.normal(0, 1, 30)
    _, p = stats.ttest_ind(group_a, group_b)
    null_p_values.append(p)

false_positives = sum(1 for p in null_p_values if p < 0.05)
print("⚠️  p-Hacking Demo: What happens when H₀ is TRUE?")
print(f"   Ran {n_experiments:,} tests where H₀ is TRUE")
print(f"   False positives (p<0.05): {false_positives} ({false_positives/n_experiments:.1%})")
print(f"   ≈ α = 5%! p<0.05 is NOT '95% sure it's real'")
print()
print("💡 If you run 100 tests with α=0.05 and H₀ true everywhere:")
print("   You'd expect ~5 'significant' results — pure luck!")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(null_p_values, bins=50, color='#3498db', edgecolor='white', density=True, alpha=0.8)
ax.axvline(0.05, color='red', lw=2.5, linestyle='--', label='α=0.05 threshold')
ax.fill_between([0, 0.05], [20, 20], alpha=0.3, color='red')
ax.set_xlabel('p-value'); ax.set_ylabel('Density')
ax.set_title('⚠️ Under H₀, p-values are UNIFORM! 5% fall below 0.05.', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('ch18_phacking.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Practice Problems

**P1:** A researcher says "p=0.04 means there is a 96% chance our hypothesis is correct."
What are the two things wrong with this statement?

**P2:** Study A: n=10, p=0.04. Study B: n=10,000, p=0.04.
Which has a larger effect size? What can you conclude from each?

**P3:** If you collect data until p<0.05, why is this a problem?

<details><summary>💡 Solutions</summary>

**P1:** (1) p-value is NOT P(H₀ true) — it's P(data | H₀ true). (2) Assumes H₀ is true; doesn't say H₁ is 96% likely.

**P2:** Study A likely has a LARGER effect (small n + small p = big effect). Study B's tiny p might reflect a trivially small effect that's meaningless in practice.

**P3:** This is optional stopping / p-hacking. The Type I error rate inflates far above α=0.05 because you're essentially running many tests and stopping when lucky.
</details>

In [ ]:
# Effect size vs statistical significance
print("Effect Size (Cohen's d) vs p-value")
for n, diff, sigma in [(10, 5, 5), (10000, 0.1, 5), (100, 2, 5)]:
    d = diff / sigma  # Cohen's d
    se = sigma / np.sqrt(n)
    t = diff / se
    p = 2 * stats.t.sf(abs(t), df=n-1)
    print(f"  n={n:>5}, diff={diff}, d={d:.2f}: p={p:.4f}, {'significant' if p<0.05 else 'not significant'}")

## 🎯 Recap
1. p-value = P(data this extreme | H₀ true). NOT P(H₀ is true).
2. Small p ≠ big effect. Large n gives small p even for tiny effects.
3. p-hacking inflates false positives — always plan your analysis before collecting data.

**Next:** [Chapter 19 — Confidence Intervals]